# 17 — Gold Churn Metrics

## Configuration

In [0]:
# Built on fact_citizen_reports, not the sensor fact tables — citizen
# reports are genuinely sporadic (real gaps happen), unlike
# fact_street_readings/fact_traffic_counts, which sample every 10 seconds
# across the identical fixed window for every street — "days since last
# reading" wouldn't discriminate between streets there at all.
#
# Anchored on MAX(message_date) from the data itself, not current_date().
# The dataset is historical (2023-06-02 to 2024-03-11) — using real
# wall-clock current_date() years later would mark every single row as
# "CRITICAL", which tells you nothing. Thresholds (30/60/120 days) are
# scaled to the dataset's own ~284-day span, not the reference project's
# 90/180/365-day thresholds sized for an open-ended real-time dataset.

from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("gold_schema", "gold", "2. Gold Schema")
CATALOG = dbutils.widgets.get("catalog_name")
GOLD = dbutils.widgets.get("gold_schema")

FACT_REPORTS = f"{CATALOG}.{GOLD}.fact_citizen_reports"
DIM_STREET = f"{CATALOG}.{GOLD}.dim_street"
STALE_TABLE = f"{CATALOG}.{GOLD}.agg_stale_streets"
TIER_TABLE = f"{CATALOG}.{GOLD}.agg_danger_tier_churn"

as_of_date = spark.sql(f"SELECT MAX(message_date) d FROM {FACT_REPORTS}").collect()[0]["d"]
print(f"As-of date (dataset's own max, not wall-clock): {as_of_date}")

## Agg Stale Streets

In [0]:
# LEFT JOIN from dim_street (all 36 active streets), not the fact table —
# streets with ZERO citizen reports must still show up, as NEVER_REPORTED,
# rather than silently disappearing because a GROUP BY on the fact alone
# would never produce a row for them.
spark.sql(f"""
CREATE OR REPLACE TABLE {STALE_TABLE}
USING DELTA
TBLPROPERTIES (
  'quality' = 'gold',
  'metric_type' = 'churn',
  'churn_grain' = 'street_id',
  'as_of_date' = '{as_of_date}'
)
AS
WITH report_activity AS (
    SELECT
        street_id,
        MAX(message_date) AS last_report_date,
        MIN(message_date) AS first_report_date,
        COUNT(DISTINCT message_date) AS active_days,
        COUNT(*) AS total_reports
    FROM {FACT_REPORTS}
    WHERE street_id IS NOT NULL
    GROUP BY street_id
)
SELECT
    d.street_id,
    d.street_name,
    ra.last_report_date,
    ra.first_report_date,
    COALESCE(ra.active_days, 0) AS active_days,
    COALESCE(ra.total_reports, 0) AS total_reports,
    DATEDIFF('{as_of_date}', ra.last_report_date) AS days_since_last_report,
    CASE
        WHEN ra.street_id IS NULL THEN 'NEVER_REPORTED'
        WHEN DATEDIFF('{as_of_date}', ra.last_report_date) > 120 THEN 'CRITICAL'
        WHEN DATEDIFF('{as_of_date}', ra.last_report_date) > 60 THEN 'STALE'
        WHEN DATEDIFF('{as_of_date}', ra.last_report_date) > 30 THEN 'AT_RISK'
        ELSE 'ACTIVE'
    END AS churn_status,
    current_timestamp() AS metric_computed_at
FROM {DIM_STREET} d
LEFT JOIN report_activity ra ON d.street_id = ra.street_id
WHERE d.__END_AT IS NULL
""")
spark.sql(f"COMMENT ON TABLE {STALE_TABLE} IS 'Churn: streets by days since last citizen report, as of {as_of_date}. Includes NEVER_REPORTED streets via LEFT JOIN from dim_street.'")

stale_count = spark.table(STALE_TABLE).count()
print(f"\nStreets scored: {stale_count} (expect 36)")
spark.table(STALE_TABLE).groupBy("churn_status").count().orderBy(F.desc("count")).show(truncate=False)


## Agg Danger Tier Churn

In [0]:
# Rollup by danger_score tier — parallels the reference's brand-level
# churn score, one level up from individual streets.

spark.sql(f"""
CREATE OR REPLACE TABLE {TIER_TABLE}
USING DELTA
TBLPROPERTIES (
  'quality' = 'gold',
  'metric_type' = 'churn',
  'churn_grain' = 'danger_tier',
  'as_of_date' = '{as_of_date}'
)
AS
WITH tiered AS (
    SELECT
        s.*,
        d.danger_score,
        CASE
            WHEN d.danger_score > 0.67 THEN 'HIGH'
            WHEN d.danger_score > 0.34 THEN 'MEDIUM'
            ELSE 'LOW'
        END AS danger_tier
    FROM {STALE_TABLE} s
    JOIN {DIM_STREET} d ON s.street_id = d.street_id AND d.__END_AT IS NULL
)
SELECT
    danger_tier,
    COUNT(*) AS total_streets,
    SUM(CASE WHEN churn_status IN ('STALE', 'CRITICAL', 'NEVER_REPORTED') THEN 1 ELSE 0 END) AS stale_or_worse_streets,
    ROUND(AVG(total_reports), 1) AS avg_reports_per_street,
    ROUND(
        SUM(CASE WHEN churn_status IN ('STALE', 'CRITICAL', 'NEVER_REPORTED') THEN 1 ELSE 0 END)
        / COUNT(*) * 100.0, 1
    ) AS churn_score_pct,
    current_timestamp() AS metric_computed_at
FROM tiered
GROUP BY danger_tier
ORDER BY churn_score_pct DESC
""")
spark.sql(f"COMMENT ON TABLE {TIER_TABLE} IS 'Brand-churn-equivalent rollup: pct of streets stale/critical/never-reported, by danger_score tier.'")

print("\nDanger-tier churn:")
display(spark.table(TIER_TABLE))